# Time sync across the vehicle -- rekon10

**Question.** For one flight, how well did the clocks on the vehicle agree, against what the
design says they should (`docs/time-base.md`)? Every later notebook places data on the FC clock
through a chain of these pairings; this one measures each link so the others can assume it.

The clocks and the records each one leaves:

| clock | record |
|---|---|
| FC `TimeUS` (free-running) | dataflash `GPS` (`GWk`/`GMS`), `RTC`, `TSYN`, `CAM` (one per PPS edge on pad S11) |
| GPS time | dataflash `GPS` week and ms |
| the DS3234 pulse | `pps.log` on the coordinator and each pod: system time at each assert |
| coordinator realtime (chrony, PPS-disciplined) | `timesync.jsonl` (FC monotonic against our realtime), `fc_time.jsonl` (UTC through the FC), `vehicle.tlog` (`SYSTEM_TIME`), collectd `chrony/*` |
| pod realtime (chrony client of the coordinator, own PPS refclock) | `pps.log`, collectd `chrony/*`, `journal.log`, `frames.jsonl` and every still's sidecar |
| ground (cluster NTP) | `ground/*.tlog` |

What comes out: one row per pairing with its offset, rate and residual over the flight, the
series behind each, and `derived/time-sync.json`. Nothing here is a verdict on the design; the
numbers sit next to the figures the design advertises and the reader compares them.

**Which coordinator session goes with which FC log** is not assumed. The FC rebooted inside a
coordinator session on at least one flight, so FC boots are found from `timesync.jsonl` (the FC's
monotonic clock restarting), each is given a UTC boot epoch, and each `.bin` is matched to a boot
by its own GPS-derived boot epoch.

In [ ]:
# --- papermill parameters ---
flight_dir = "/home/jovyan/datasets/flights/rekon10/2026-10-25-soft-mount-check-flight"
captures_dir = ""   # "" -> <flight_dir>/captures ; sessions as captures/<node>/<session>/
ground_dir = ""     # "" -> <flight_dir>/ground
out_dir = ""        # "" -> <flight_dir>/derived
fc_bin = ""         # "" -> every *.bin at the flight root or under fc/


In [ ]:
import json, re, os, sys, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymavlink import mavutil
REPO = Path.cwd()
while not (REPO / "analysis" / "ardupilot_log.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "analysis"))
import tlog_read

FLIGHT = Path(flight_dir)
CAP = Path(captures_dir) if captures_dir else FLIGHT / "captures"
GND = Path(ground_dir) if ground_dir else FLIGHT / "ground"
OUT = Path(out_dir) if out_dir else FLIGHT / "derived"
OUT.mkdir(parents=True, exist_ok=True)
BINS = [Path(fc_bin)] if fc_bin else sorted(FLIGHT.glob("*.bin")) + sorted((FLIGHT / "fc").glob("*.bin"))

GPS_EPOCH = 315964800          # 1980-01-06 as unix
GPS_UTC_LEAP_S = 18            # GPS - UTC; ArduPilot's GPS_LEAPSECONDS default
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
REPORT = {"flight": str(FLIGHT), "pairings": [], "fc_logs": {}, "machines": {}}

def jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except json.JSONDecodeError: pass
    return pd.DataFrame(rows)

def load_pps(path):
    # ppstest lines: "... assert 1792938361.657837324, sequence: 18 - clear ..."
    rx = re.compile(r"^(\S+)\s.*assert (\d+\.\d+), sequence: (\d+)")
    rows = []
    for line in open(path, errors="replace"):
        m = rx.match(line)
        if m:
            rows.append((m.group(1), float(m.group(2)), int(m.group(3))))
    df = pd.DataFrame(rows, columns=["journal_stamp", "t", "seq"])
    if len(df):
        df["phase_us"] = (df.t - np.round(df.t)) * 1e6     # system time minus the nearest second
        df["dt"] = df.t.diff()
        df["dseq"] = df.seq.diff()
    return df

def load_chrony(d):
    # collectd csv plugin: <metric>-<instance>-<YYYY-MM-DD>, rows "epoch,value"; a boot whose
    # clock stepped leaves two dates for one series, so every file of a series is concatenated.
    rx = re.compile(r"^(?P<metric>[a-z_]+)-(?P<inst>.+)-(\d{4}-\d{2}-\d{2})$")
    series = {}
    for f in sorted(Path(d).glob("*")):
        m = rx.match(f.name)
        if not m: continue
        try:
            df = pd.read_csv(f)
        except Exception:
            continue
        if not {"epoch", "value"} <= set(df.columns): continue
        key = (m["metric"], m["inst"])
        series[key] = pd.concat([series.get(key, pd.DataFrame()), df]).sort_values("epoch")
    return series

def gps_unix(gwk, gms):
    return GPS_EPOCH + gwk * 604800.0 + gms / 1000.0 - GPS_UTC_LEAP_S

def fit_line(x, y, robust=False):
    # y = a + b x ; returns a, b, residual rms, n. x and y in seconds. robust: one pass of
    # rejecting residuals beyond 5x the median absolute residual, for link-delay outliers.
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 2: return None
    b, a = np.polyfit(x, y, 1)
    r = y - (a + b * x)
    keep = np.ones(len(x), bool)
    if robust and len(x) >= 4:
        mad = np.median(np.abs(r - np.median(r)))
        keep = np.abs(r - np.median(r)) <= max(5 * mad, 1e-6)
        if keep.sum() >= 3 and keep.sum() < len(x):
            b, a = np.polyfit(x[keep], y[keep], 1)
            r = y - (a + b * x)
    return dict(a=float(a), b=float(b), rms=float(np.sqrt(np.mean(r[keep]**2))), n=int(keep.sum()), n_rejected=int((~keep).sum()), resid=r, keep=keep)

LOCK_US = 1000.0   # a clock is "locked" to the pulse once every later assert sits within this of the second

def pps_lock(pps):
    # Convergence after boot: the first assert after which every later |phase| < LOCK_US. Before it
    # chrony is stepping or slewing (its default maxslewrate is 83333 ppm, 83 ms per second, which
    # is the slope the phase walks at). Returns the lock time, how long that took, the slew seen.
    if not len(pps): return None
    off = pps.phase_us.abs().values > LOCK_US
    last_bad = np.where(off)[0].max() if off.any() else -1
    lock_i = last_bad + 1
    if lock_i >= len(pps): return dict(locked=False, n_unlocked=int(off.sum()), max_abs_us=float(pps.phase_us.abs().max()))
    t_lock = float(pps.t.iloc[lock_i])
    conv = pps.iloc[:lock_i + 1]       # up to and including the first locked assert, whose dt is the step when the step is what locked it
    # a clock step (chrony "stepped by N seconds") shows as an assert interval far from 1 s; the
    # time to lock is counted from the last such step, or from the first assert if there was none
    steps = conv[(conv.dt - 1).abs() > 3600]
    t_ref = float(steps.t.iloc[-1]) if len(steps) else float(pps.t.iloc[0])
    slew = float(np.nanmax(np.abs(conv.phase_us.diff()))) / 1e3 if len(conv) > 1 else 0.0
    return dict(locked=True, t_first=float(pps.t.iloc[0]), t_lock=t_lock, s_to_lock=float(t_lock - t_ref),
                clock_steps_s=[float(v - 1) for v in steps.dt], n_unlocked=int(off.sum()),
                max_abs_us=float(pps.phase_us.abs().max()), max_phase_change_ms_per_s=slew)

def phase_stats(pps, t0=None, t1=None):
    p = pps if t0 is None else pps[(pps.t >= t0) & (pps.t <= t1)]
    if not len(p): return dict(n=0)
    return dict(n=int(len(p)), median_us=float(p.phase_us.median()), sd_us=float(p.phase_us.std()), max_abs_us=float(p.phase_us.abs().max()),
                missing_seconds=int(np.nansum(np.maximum(p.dseq - 1, 0))), interval_max_s=float(p.dt.max()))

def sessions_of(kind):
    # captures/<node>/<session>/ ; kind "coordinator" or "campod"
    out = []
    if not CAP.exists(): return out
    for node in sorted(CAP.iterdir()):
        if not node.is_dir(): continue
        if kind == "coordinator" and node.name != "coordinator": continue
        if kind == "campod" and not node.name.startswith("campod"): continue
        for s in sorted(node.iterdir()):
            if s.is_dir() and not s.name.startswith("@"):
                out.append(s)
    return out

print("flight:", FLIGHT.name)
print("captures:", CAP, "exists" if CAP.exists() else "MISSING")
print("FC logs:", [b.name for b in BINS])


## 1. What this flight recorded

Every machine, every session, every time record and its size. Absence is a result here: a pps.log
with no asserts, a `.bin` with no `CAM` rows, an empty ground tlog each say something about the
flight, and each shows up below as a count of zero rather than as a missing row.

In [ ]:
inv = []
for b in BINS:
    inv.append(dict(machine="fc", session=b.name, record=".bin", n=b.stat().st_size, note="bytes"))
for s in sessions_of("coordinator") + sessions_of("campod"):
    node = s.parent.name
    for name in ("pps.log", "timesync.jsonl", "fc_time.jsonl", "vehicle.tlog", "frames.jsonl", "journal.log"):
        f = s / name
        if f.exists():
            if name == "pps.log":
                n = sum(1 for l in open(f, errors="replace") if " assert " in l); note = "asserts"
            elif name.endswith(".jsonl"):
                n = sum(1 for l in open(f) if l.strip()); note = "rows"
            elif name == "journal.log":
                n = sum(1 for l in open(f, errors="replace") if "chronyd" in l); note = "chronyd lines"
            else:
                n = f.stat().st_size; note = "bytes"
        else:
            n, note = 0, "absent"
        inv.append(dict(machine=node, session=s.name[:8], record=name, n=n, note=note))
    stills = [p for p in s.glob("*.json") if p.name.startswith(node + "_")]
    inv.append(dict(machine=node, session=s.name[:8], record="still sidecars", n=len(stills), note="files"))
    ch = s / "collectd" / node / "chrony"
    inv.append(dict(machine=node, session=s.name[:8], record="collectd chrony", n=len(list(ch.glob("*"))) if ch.exists() else 0, note="series files"))
for f in sorted(GND.glob("*")) if GND.exists() else []:
    inv.append(dict(machine="ground", session="", record=f.name, n=f.stat().st_size, note="bytes"))
inv = pd.DataFrame(inv)
display(inv)
REPORT["inventory"] = inv.to_dict("records")


## 2. The FC against GPS, and what else the FC logged about time

`GPS` rows with a week number give UTC at each `TimeUS`; a line fit gives the FC clock's rate
against GPS and the residual, and its boot epoch (UTC at `TimeUS` 0). `RTC` rows are the FC's own
UTC mapping, which it gets from GPS and later hands to the DS3234 while disarmed. `TSYN` is the
round trip the FC measured on its own TIMESYNC requests. `CAM` rows are PPS edges captured on pad
S11: with any present, the FC's rate against the pulse comes straight from them.

In [ ]:
FC = {}
for b in BINS:
    rows = {t: [] for t in ("GPS", "RTC", "TSYN", "CAM", "ARM", "EV", "PARM", "MAV")}
    m = mavutil.mavlink_connection(str(b), robust_parsing=True)
    while True:
        x = m.recv_match(type=list(rows), blocking=False)
        if x is None: break
        rows[x.get_type()].append(x.to_dict())
    F = {t: pd.DataFrame(v) for t, v in rows.items() if v}
    parms = {r.Name: r.Value for r in F["PARM"].itertuples()} if "PARM" in F else {}
    info = dict(name=b.name, n_gps=len(F.get("GPS", [])), n_rtc=len(F.get("RTC", [])),
                n_tsyn=len(F.get("TSYN", [])), n_cam=len(F.get("CAM", [])),
                cam1_type=parms.get("CAM1_TYPE"), cam1_feedbak_pin=parms.get("CAM1_FEEDBAK_PIN"))
    g = F.get("GPS", pd.DataFrame())
    g = g[(g.get("GWk", 0) > 1000)] if len(g) else g
    fit = fit_line(g.TimeUS / 1e6, gps_unix(g.GWk.values, g.GMS.values)) if len(g) >= 2 else None
    if fit:
        # rate convention everywhere below: A vs B, positive = A runs fast relative to B
        info.update(gps_fit=dict(boot_utc=fit["a"], rate_ppm=(1 / fit["b"] - 1) * 1e6, rms_ms=fit["rms"] * 1e3, n=fit["n"],
                                 span_s=[float(g.TimeUS.min() / 1e6), float(g.TimeUS.max() / 1e6)]))
    r = F.get("RTC", pd.DataFrame())
    if len(r) and fit:
        r = r[r.Epoch > 0]
        if len(r):
            d = r.Epoch / 1e6 - (fit["a"] + fit["b"] * r.TimeUS / 1e6)
            info["rtc_minus_gps_ms"] = dict(median=float(np.median(d) * 1e3), first=float(d.iloc[0] * 1e3), last=float(d.iloc[-1] * 1e3),
                                             max_abs=float(np.abs(d).max() * 1e3), n=int(len(d)),
                                             span_s=[float(r.TimeUS.min() / 1e6), float(r.TimeUS.max() / 1e6)],
                                             sources=sorted(set(int(v) for v in r.SourceType)))
            info["rtc_boot_utc"] = float(np.median(r.Epoch / 1e6 - r.TimeUS / 1e6))
    if "TSYN" in F:
        info["tsyn_rtt_us"] = dict(median=float(F["TSYN"].RTT.median()), max=float(F["TSYN"].RTT.max()), n=int(len(F["TSYN"])))
    if "CAM" in F and fit:
        c = F["CAM"]
        tcol = "TimeUS"
        ut = fit["a"] + fit["b"] * c[tcol] / 1e6
        info["cam_edges"] = dict(n=int(len(c)), gps_phase_ms=dict(median=float(np.median((ut - np.round(ut)) * 1e3)),
                                                                    sd=float(np.std((ut - np.round(ut)) * 1e3))))
        ef = fit_line(np.round(ut), c[tcol] / 1e6)
        if ef: info["cam_edges"]["fc_rate_vs_pulse_ppm"] = (ef["b"] - 1) * 1e6; info["cam_edges"]["rms_us"] = ef["rms"] * 1e6   # x = pulse seconds, y = FC: b-1 is FC vs pulse
    arm = F.get("ARM", pd.DataFrame())
    if len(arm):
        info["arm_events"] = [dict(t_s=float(t / 1e6), armed=int(s)) for t, s in zip(arm.TimeUS, arm.ArmState)]
    FC[b.name] = dict(info=info, F=F, fit=fit)
    REPORT["fc_logs"][b.name] = info
    print(json.dumps(info, indent=1, default=str))


In [ ]:
if not FC:
    print("no DataFlash log in this flight directory: the FC rows below are empty and the pairings that need the FC are skipped "
          "(a bench cycle has no FC log; the pods' and coordinator's own records are still measured)")
fig, axes = plt.subplots(max(len(FC), 1), 1, figsize=(11, 2.2 * max(len(FC), 1)), squeeze=False)
for ax, (name, d) in zip(axes[:, 0], FC.items()):
    if d["fit"]:
        g = d["F"]["GPS"]; g = g[g.GWk > 1000]
        ax.plot(g.TimeUS / 1e6, d["fit"]["resid"] * 1e3, ".", ms=2)
        ax.set_title(f"{name}: FC TimeUS vs GPS, residual after a line fit  (rate {d['info']['gps_fit']['rate_ppm']:+.1f} ppm, rms {d['info']['gps_fit']['rms_ms']:.1f} ms)")
        ax.set_ylabel("ms")
    else:
        ax.set_title(f"{name}: no GPS rows with a week number")
    for e in d["info"].get("arm_events", []):
        ax.axvline(e["t_s"], color="k" if e["armed"] else "grey", lw=0.8)
axes[-1, 0].set_xlabel("FC TimeUS (s)")
fig.tight_layout(); fig.savefig(OUT / "time-sync-fc-vs-gps.png"); plt.show()


## 3. The coordinator: the pulse, the FC's clock against ours, and chrony

`pps.log` is the system time at every assert; its fractional part is the clock's phase against the
pulse, and a step in it is a re-phase. `timesync.jsonl` pairs the FC's monotonic clock with our
realtime at each exchange; segmenting on the FC's clock restarting finds FC boots, and a line fit
per boot gives the FC's rate against the coordinator's PPS-disciplined clock, with no GPS in the
path. `fc_time.jsonl` is this box's UTC offset measured through the FC (`SYSTEM_TIME` composed
with TIMESYNC). The chrony series are collectd's view of which source was selected and how far
off each was.

In [ ]:
def chrony_summary(ch, inst, t_lock=None):
    # Post-lock figures where a lock time is known (the convergence after boot is reported by
    # pps_lock, not averaged in here); max_abs_all is over the whole session.
    out = {}
    for metric in ("time_offset", "clock_skew_ppm", "clock_reachability", "clock_state", "clock_stratum", "frequency_error"):
        s = ch.get((metric, inst))
        if s is None: continue
        s = s.dropna(subset=["value"])
        if not len(s): out[metric] = dict(n=0); continue
        v_all = s.value
        v = s[s.epoch >= t_lock].value if t_lock is not None else v_all
        if metric in ("clock_state", "clock_stratum", "clock_reachability"):
            out[metric] = dict(n=int(len(v)), values=sorted(set(float(x) for x in v)), last=float(v_all.iloc[-1]))
        else:
            out[metric] = dict(n=int(len(v)), median=float(v.median()) if len(v) else None, max_abs=float(v.abs().max()) if len(v) else None,
                               last=float(v_all.iloc[-1]), max_abs_all=float(v_all.abs().max()))
    return out

def pps_block(pps):
    d = dict(n=int(len(pps)), t_first=float(pps.t.min()), t_last=float(pps.t.max()),
             interval_s=dict(median=float(pps.dt.median()), min=float(pps.dt.min()), max=float(pps.dt.max())),
             seq_gaps=int((pps.dseq > 1).sum()), missing_seconds=int(np.nansum(np.maximum(pps.dseq - 1, 0))),
             lock=pps_lock(pps))
    if d["lock"] and d["lock"].get("locked"):
        d["locked_phase"] = phase_stats(pps, d["lock"]["t_lock"], pps.t.max())
    return d

COORD = {}
for s in sessions_of("coordinator"):
    d = dict(session=s.name)
    pps = load_pps(s / "pps.log") if (s / "pps.log").exists() else pd.DataFrame()
    t_lock = None
    if len(pps):
        d["pps"] = pps_block(pps)
        t_lock = (d["pps"]["lock"] or {}).get("t_lock")
    ts = jsonl(s / "timesync.jsonl") if (s / "timesync.jsonl").exists() else pd.DataFrame()
    boots = []
    if len(ts):
        ts = ts.sort_values("tc1_realtime_ns").reset_index(drop=True)
        ts["boot"] = (ts.fc_ts1_ns.diff() < 0).cumsum()
        # pairs taken while the coordinator's own clock was still converging are not pairs of
        # two steady clocks; they are left out of the rate fit and counted.
        ts["steady"] = (ts.tc1_realtime_ns / 1e9 >= t_lock) if t_lock else True
        for bi, grp in ts.groupby("boot"):
            boot_utc = (grp.tc1_realtime_ns - grp.fc_ts1_ns).median() / 1e9
            g2 = grp[grp.steady]
            fit = fit_line(g2.fc_ts1_ns / 1e9, g2.tc1_realtime_ns / 1e9, robust=True) if len(g2) >= 3 else None
            boots.append(dict(boot=int(bi), n=int(len(grp)), n_steady=int(len(g2)), boot_utc=float(boot_utc),
                              fc_rate_vs_coord_ppm=((1 / fit["b"] - 1) * 1e6) if fit else None,     # x = FC, y = realtime
                              rms_ms=(fit["rms"] * 1e3) if fit else None, n_fit=fit["n"] if fit else 0,
                              span_s=float((grp.fc_ts1_ns.max() - grp.fc_ts1_ns.min()) / 1e9)))
    d["timesync_boots"] = boots
    ft = jsonl(s / "fc_time.jsonl") if (s / "fc_time.jsonl").exists() else pd.DataFrame()
    segs = []
    if len(ft):
        ft = ft.sort_values("local_us").reset_index(drop=True)
        ft["boot"] = (ft.fc_boot_us.diff() < 0).cumsum()           # the FC's monotonic restarting = a new FC boot
        ft["steady"] = (ft.local_us / 1e6 >= t_lock) if t_lock else True
        for bi, grp in ft.groupby("boot"):
            f2 = grp[grp.steady]
            seg = dict(boot=int(bi), n=int(len(grp)), n_steady=int(len(f2)),
                       boot_utc=float(((grp.local_us + grp.offset_us) - grp.fc_boot_us).median() / 1e6),   # coordinator realtime at FC TimeUS 0
                       fc_minus_coord_ms=dict(first=float(grp.offset_us.iloc[0] / 1e3), median=float(f2.offset_us.median() / 1e3) if len(f2) else None,
                                              last=float(grp.offset_us.iloc[-1] / 1e3)),
                       rtt_ms=dict(median=float(grp.rtt_us.median() / 1e3), max=float(grp.rtt_us.max() / 1e3)))
            if len(f2) >= 3:
                sl = fit_line(f2.local_us / 1e6, f2.offset_us / 1e6)
                seg["fc_minus_coord_drift_ppm"] = sl["b"] * 1e6; seg["fit_rms_ms"] = sl["rms"] * 1e3
                # the FC's own UTC-minus-monotonic (SYSTEM_TIME), as the router recorded it: a boot
                # epoch by the FC's RTC, and how it moved over the session
                umb = (f2.fc_unix_us - f2.fc_boot_us) / 1e6
                seg["fc_rtc_boot_utc"] = dict(median=float(umb.median()), first=float(umb.iloc[0]), last=float(umb.iloc[-1]))
                # the coordinator-initiated exchange gives the FC's monotonic directly (fc_boot_us), a
                # second rate measurement of the FC against the coordinator
                sl2 = fit_line(f2.local_us / 1e6, f2.fc_boot_us / 1e6)
                seg["fc_rate_vs_coord_ppm"] = (sl2["b"] - 1) * 1e6; seg["fc_rate_fit_rms_ms"] = sl2["rms"] * 1e3
            segs.append(seg)
    d["fc_time_boots"] = segs
    chd = s / "collectd" / "coordinator" / "chrony"
    ch = load_chrony(chd) if chd.exists() else {}
    d["chrony"] = {inst: chrony_summary(ch, inst, t_lock) for inst in sorted({k[1] for k in ch})}
    COORD[s.name] = dict(info=d, pps=pps, ts=ts, ft=ft, ch=ch, t_lock=t_lock)
    d.pop("fc_time", None)
    REPORT["machines"][f"coordinator/{s.name}"] = d
    print(json.dumps(d, indent=1, default=str))


### Pairing FC logs with coordinator sessions and FC boots

A `.bin`'s boot epoch comes from its GPS fit; a boot seen by the coordinator has one from
`timesync.jsonl`. They match when they agree to within a few seconds, which is the GPS-fit
extrapolation to `TimeUS` 0 plus the TIMESYNC one-way delay; nothing finer is needed to pair.

In [ ]:
PAIR = {}
for name, d in FC.items():
    best = None
    if d["fit"]:
        for sname, c in COORD.items():
            for b in c["info"]["timesync_boots"]:
                err = abs(b["boot_utc"] - d["fit"]["a"])
                if best is None or err < best[2]:
                    best = (sname, b, err)
    if best and best[2] < 10:
        PAIR[name] = dict(session=best[0], boot=best[1]["boot"], boot_epoch_diff_s=float(best[2]), timesync=best[1])
        fts = [sg for sg in COORD[best[0]]["info"]["fc_time_boots"] if abs(sg["boot_utc"] - d["fit"]["a"]) < 10]
        PAIR[name]["fc_time"] = fts[0] if fts else None
    else:
        PAIR[name] = dict(session=None, note="no coordinator boot within 10 s of this log's boot epoch" if d["fit"] else "no GPS fit")
    REPORT["fc_logs"][name]["pairing"] = PAIR[name]
pd.DataFrame([dict(fc_log=k, **{kk: vv for kk, vv in v.items() if kk != "timesync"},
                   fc_rate_vs_coord_ppm=(v.get("timesync") or {}).get("fc_rate_vs_coord_ppm"),
                   timesync_rms_ms=(v.get("timesync") or {}).get("rms_ms"),
                   fc_rate_vs_gps_ppm=(FC[k]["info"].get("gps_fit") or {}).get("rate_ppm")) for k, v in PAIR.items()])


### The FC's rate: against GPS, against the pulse, and the pulse against GPS

Rates are *A vs B, positive = A runs fast relative to B*, in ppm.

The FC's clock is one oscillator measured against GPS from the dataflash, and against the
coordinator's PPS-disciplined realtime two ways: the FC's own TIMESYNC requests (`timesync.jsonl`,
one-way delay included) and the coordinator's requests (`fc_time.jsonl`, round trip halved). The
difference between the FC-vs-GPS and FC-vs-coordinator rates is the coordinator's clock, so the
pulse, against GPS. With `CAM` edges in the log the FC-against-pulse figure comes from hardware
capture instead; where both exist they are both listed.

A second table puts the FC's boot epoch -- UTC at `TimeUS` 0 -- from each source side by side:
the GPS fit, the `RTC` rows, `SYSTEM_TIME` as the router recorded it, and TIMESYNC. The GPS fit is
the reference; the others' differences from it are in ms.

In [ ]:
tri = []; epochs = []
for name, p in PAIR.items():
    gf = FC[name]["info"].get("gps_fit")
    if not gf: continue
    row = dict(fc_log=name, fc_vs_gps_ppm=gf["rate_ppm"], gps_rms_ms=gf["rms_ms"])
    ep = dict(fc_log=name, gps_fit=gf["boot_utc"])
    if FC[name]["info"].get("rtc_boot_utc"): ep["rtc_rows_minus_gps_ms"] = (FC[name]["info"]["rtc_boot_utc"] - gf["boot_utc"]) * 1e3
    tsb = p.get("timesync")
    if tsb and tsb.get("fc_rate_vs_coord_ppm") is not None:
        row.update(fc_vs_coord_ppm_timesync=tsb["fc_rate_vs_coord_ppm"], timesync_rms_ms=tsb["rms_ms"], timesync_n=tsb["n_fit"],
                   coord_vs_gps_ppm_via_timesync=gf["rate_ppm"] - tsb["fc_rate_vs_coord_ppm"])
        ep["timesync_minus_gps_ms"] = (tsb["boot_utc"] - gf["boot_utc"]) * 1e3
    ftd = p.get("fc_time")
    if ftd and ftd.get("fc_rate_vs_coord_ppm") is not None:
        row.update(fc_vs_coord_ppm_fc_time=ftd["fc_rate_vs_coord_ppm"], fc_time_rms_ms=ftd["fc_rate_fit_rms_ms"],
                   coord_vs_gps_ppm_via_fc_time=gf["rate_ppm"] - ftd["fc_rate_vs_coord_ppm"])
        ep["system_time_minus_gps_ms"] = (ftd["fc_rtc_boot_utc"]["median"] - gf["boot_utc"]) * 1e3
        ep["system_time_first_to_last_ms"] = (ftd["fc_rtc_boot_utc"]["last"] - ftd["fc_rtc_boot_utc"]["first"]) * 1e3
    ce = FC[name]["info"].get("cam_edges")
    if ce and "fc_rate_vs_pulse_ppm" in ce:
        row.update(fc_vs_pulse_ppm_cam=ce["fc_rate_vs_pulse_ppm"], cam_rms_us=ce["rms_us"], cam_n=ce["n"])
    tri.append(row); epochs.append(ep)
tri = pd.DataFrame(tri); epochs = pd.DataFrame(epochs)
display(tri); display(epochs)
REPORT["fc_rate_triangle"] = tri.to_dict("records"); REPORT["fc_boot_epochs"] = epochs.to_dict("records")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5))
for sname, c in COORD.items():
    if len(c["pps"]):
        axes[0].plot(c["pps"].t, c["pps"].phase_us, ".", ms=2, label=f"coordinator {sname[:8]}")
    ts = c["ts"]
    if len(ts):
        for bi, grp in ts[ts.steady].groupby("boot"):
            fit = fit_line(grp.fc_ts1_ns / 1e9, grp.tc1_realtime_ns / 1e9, robust=True) if len(grp) >= 3 else None
            if fit:
                axes[1].plot(grp.tc1_realtime_ns / 1e9, fit["resid"] * 1e3, "o-", ms=3, label=f"{sname[:8]} FC boot {bi}: {(fit['b']-1)*1e6:+.0f} ppm, rms {fit['rms']*1e3:.2f} ms, {fit['n_rejected']} rejected")
                axes[1].plot(grp.tc1_realtime_ns[~fit["keep"]] / 1e9, fit["resid"][~fit["keep"]] * 1e3, "rx", ms=6)
axes[0].set_ylabel("PPS assert phase (us)"); axes[0].set_title("coordinator: system time at each PPS assert, minus the nearest second")
axes[0].legend(fontsize=7); axes[0].set_ylim(-1500, 1500)
axes[1].set_ylabel("ms"); axes[1].set_title("timesync.jsonl: FC monotonic vs coordinator realtime, residual per FC boot (after the coordinator's clock locked)"); axes[1].legend(fontsize=7)
axes[1].set_ylim(-20, 20)
axes[1].set_xlabel("unix s")
fig.tight_layout(); fig.savefig(OUT / "time-sync-coordinator.png"); plt.show()


## 4. The pods: the pulse, chrony against the coordinator, and the frames

Each pod logs its own asserts, runs chrony against the coordinator over the gadget link with its
own PPS refclock, and stamps each frame. `frames.jsonl` carries `second_offset_us`, the exposure's
position inside its second, which is what the sync design asks every pod to hold in common.
`journal.log` has chrony's own account: the step at boot, which source it selected, and when it
could not.

In [ ]:
PODS = {}
for s in sessions_of("campod"):
    node = s.parent.name
    d = dict(node=node, session=s.name)
    pps = load_pps(s / "pps.log") if (s / "pps.log").exists() else pd.DataFrame()
    t_lock = None
    if len(pps):
        d["pps"] = pps_block(pps)
        t_lock = (d["pps"]["lock"] or {}).get("t_lock")
        # where the phase sat before lock, in 1 ms bins: a pod that held an offset for a while shows it
        pre = pps[pps.t < t_lock] if t_lock else pps
        d["pps"]["prelock_phase_clusters_ms"] = {str(k): int(v) for k, v in np.round(pre.phase_us / 1000).value_counts().head(6).items()}
    chd = s / "collectd" / node / "chrony"
    ch = load_chrony(chd) if chd.exists() else {}
    d["chrony"] = {inst: chrony_summary(ch, inst, t_lock) for inst in sorted({k[1] for k in ch})}
    jl = s / "journal.log"
    ev = []
    if jl.exists():
        for line in open(jl, errors="replace"):
            if "chronyd" in line and any(k in line for k in ("stepped", "Selected source", "Can't synchronise", "wrong by", "Frequency")):
                ev.append(line.strip()[:160])
    d["chrony_journal"] = ev
    fr = jsonl(s / "frames.jsonl") if (s / "frames.jsonl").exists() else pd.DataFrame()
    if len(fr) and "second_offset_us" in fr:
        kept = fr[fr.decision == "kept"] if "decision" in fr else fr
        d["frames"] = dict(n=int(len(fr)), n_keep=int(len(kept)),
                           second_offset_us=dict(median=float(fr.second_offset_us.median()), sd=float(fr.second_offset_us.std()),
                                                 min=float(fr.second_offset_us.min()), max=float(fr.second_offset_us.max())),
                           decisions=fr.decision.value_counts().to_dict() if "decision" in fr else None)
        if "clk" in fr:
            clk = pd.DataFrame(list(fr.clk))
            wmb = (clk.wall_ns - clk.boot_ns) / 1e9
            d["frames"]["wall_minus_boot_s"] = dict(first=float(wmb.iloc[0]), last=float(wmb.iloc[-1]), max_step=float(wmb.diff().abs().max()))
    sc = []
    for p in sorted(s.glob(node + "_*.json")):
        try:
            j = json.load(open(p))
        except Exception:
            continue
        if "exposure_wall_ns" in j:
            sc.append(dict(name=p.stem, seq=j.get("seq"), exposure_wall_ns=j["exposure_wall_ns"], exposure_us=j.get("exposure_us"),
                           frame_wall_clock_ns=j.get("frame_wall_clock_ns"), gain=j.get("analogue_gain", j.get("gain"))))
    sc = pd.DataFrame(sc)
    if len(sc):
        sc["phase_us"] = (sc.exposure_wall_ns % 1_000_000_000) / 1e3
        d["stills"] = dict(n=int(len(sc)), exposure_phase_us=dict(median=float(sc.phase_us.median()), sd=float(sc.phase_us.std())),
                           t_first=float(sc.exposure_wall_ns.min() / 1e9), t_last=float(sc.exposure_wall_ns.max() / 1e9))
    PODS[node] = dict(info=d, pps=pps, ch=ch, frames=fr, stills=sc, t_lock=t_lock)
    REPORT["machines"][f"{node}/{s.name}"] = d
    print(json.dumps(d, indent=1, default=str))


### The OAK-D on the coordinator, when oak-capture ran

The coordinator's own camera is not part of the pod sync group; its frames run at their own
cadence. What its sidecars give here: how many frames of each kind, the exposure and ISO they
ran at, the ledger's decisions, where in its second each colour frame sat, and the colour
lens position over the session (the gen1's autofocus is left on).

In [ ]:
OAK = {}
for s in sessions_of("coordinator"):
    rows = []
    for p in sorted(s.glob("*_color_*.json")) + sorted(s.glob("*_left_*.json")) + sorted(s.glob("*_right_*.json")):
        try: j = json.load(open(p))
        except Exception: continue
        rows.append(dict(kind=j.get("kind"), seq=j.get("seq"), device_seq=j.get("device_seq"), wall_ns=j.get("wall_ns"),
                         monotonic_ns=j.get("monotonic_ns"), exposure_us=j.get("exposure_us"), iso=j.get("iso"), lens_position=j.get("lens_position")))
    if not rows: continue
    df = pd.DataFrame(rows)
    led = [f for f in s.glob("*.frames.jsonl")]
    ledger = jsonl(led[0]) if led else pd.DataFrame()
    info = dict(session=s.name, kept_by_kind=df.kind.value_counts().to_dict(),
                exposure_us={k: sorted(set(int(v) for v in g.exposure_us.dropna()))[:6] for k, g in df.groupby("kind")},
                iso={k: dict(min=int(g.iso.min()), max=int(g.iso.max())) for k, g in df.groupby("kind") if g.iso.notna().any()},
                ledger_decisions=ledger.decision.value_counts().to_dict() if "decision" in ledger else None)
    col = df[df.kind == "color"].dropna(subset=["wall_ns"])
    if len(col):
        ph = (col.wall_ns % 1_000_000_000) / 1e3
        info["color_phase_in_second_us"] = dict(median=float(ph.median()), sd=float(ph.std()))
        dseq = col.device_seq.diff().dropna()
        info["color_device_seq_gaps"] = int((dseq > 1).sum())
        if col.lens_position.notna().any():
            lp = col.lens_position
            info["color_lens_position"] = dict(first=float(lp.iloc[0]), last=float(lp.iloc[-1]), distinct=int(lp.nunique()), min=float(lp.min()), max=float(lp.max()))
    OAK[s.name] = dict(info=info, frames=df)
    REPORT["machines"].setdefault(f"coordinator/{s.name}", {})["oak"] = info
    print(json.dumps(info, indent=1, default=str))
if not OAK: print("no OAK sidecars in any coordinator session")


### Pod against pod: the same second, the same instant?

Stills from two pods are matched by nearest `exposure_wall_ns`; the difference is how far apart
the two pods placed the same second's exposure, on their own clocks. The design figure from
flight-platform is 52 us at `buffer_count=4`.

In [ ]:
pairs = []
names = sorted(PODS)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = PODS[names[i]]["stills"], PODS[names[j]]["stills"]
        if not len(a) or not len(b): continue
        ta = np.sort(a.exposure_wall_ns.values); tb = np.sort(b.exposure_wall_ns.values)
        k = np.searchsorted(tb, ta); k = np.clip(k, 1, len(tb) - 1)
        near = np.where(np.abs(tb[k] - ta) < np.abs(tb[k - 1] - ta), tb[k], tb[k - 1])
        dt = (near - ta) / 1e3
        ok = np.abs(dt) < 500_000
        row = dict(a=names[i], b=names[j], n_matched=int(ok.sum()), n_a=int(len(ta)), n_b=int(len(tb)),
                   dt_us=dict(median=float(np.median(dt[ok])), sd=float(np.std(dt[ok])), max_abs=float(np.abs(dt[ok]).max())) if ok.any() else None)
        # the same pairing by sequence number on FrameWallClock (libcamera's own stamp), which is
        # how flight-platform measured it; both are reported so the two methods can be compared
        if "frame_wall_clock_ns" in a and "frame_wall_clock_ns" in b:
            m = a[["seq", "frame_wall_clock_ns"]].merge(b[["seq", "frame_wall_clock_ns"]], on="seq", suffixes=("_a", "_b")).dropna()
            if len(m):
                d2 = (m.frame_wall_clock_ns_b - m.frame_wall_clock_ns_a) / 1e3
                row["by_seq_frame_wall_clock_us"] = dict(n=int(len(m)), median=float(d2.median()), sd=float(d2.std()), max_abs=float(d2.abs().max()))
        pairs.append(row)
pairs = pd.DataFrame(pairs); display(pairs)
REPORT["pod_pairs"] = pairs.to_dict("records")


In [ ]:
n = max(1, len(PODS))
fig, axes = plt.subplots(3, 1, figsize=(11, 8))
for node, p in PODS.items():
    if len(p["pps"]):
        axes[0].plot(p["pps"].t, p["pps"].phase_us, ".", ms=2, label=node)
    for inst in ("PPS", "10.55.0.1"):
        s = p["ch"].get(("time_offset", inst))
        if s is not None:
            axes[1].plot(s.epoch, s.value * 1e6, ".-", ms=3, label=f"{node} chrony offset vs {inst}")
    if len(p["frames"]) and "second_offset_us" in p["frames"]:
        fr = p["frames"]
        axes[2].plot(fr.exposure_wall_ns / 1e9, fr.second_offset_us, ".", ms=3, label=f"{node} frames.jsonl second_offset_us")
    if len(p["stills"]):
        axes[2].plot(p["stills"].exposure_wall_ns / 1e9, p["stills"].phase_us, "x", ms=4, label=f"{node} kept stills")
axes[0].set_ylabel("PPS assert phase (us)"); axes[0].set_ylim(-1500, 1500); axes[0].set_title("pods: system time at each PPS assert, minus the nearest second"); axes[0].legend(fontsize=7)
axes[1].set_ylabel("us"); axes[1].set_yscale("symlog", linthresh=10); axes[1].set_title("pods: chrony time_offset per source (collectd)"); axes[1].legend(fontsize=7)
axes[2].set_ylabel("us into the second"); axes[2].set_title("pods: where in its second each exposure sat"); axes[2].legend(fontsize=7); axes[2].set_xlabel("unix s")
fig.tight_layout(); fig.savefig(OUT / "time-sync-pods.png"); plt.show()


### Which FC log each pod session belongs to

A pod session is a boot and can outlive several FC boots. The `.bin` whose GPS-dated span contains
the session's exposures is the one its stills and accelerometer streams go with; the runner reads
this from `time-sync.json` to choose the `.bin` for the per-pod notebooks.

In [ ]:
REPORT["pod_session_fc_log"] = {}
for node, p in PODS.items():
    sess = p["info"]["session"]
    times = []
    if len(p["stills"]): times += list(p["stills"].exposure_wall_ns / 1e9)
    if len(p["frames"]) and "exposure_wall_ns" in p["frames"]: times += list(p["frames"].exposure_wall_ns.dropna() / 1e9)
    if not times:
        REPORT["pod_session_fc_log"][f"{node}/{sess}"] = dict(fc_log=None, note="no exposures")
        continue
    t0, t1 = min(times), max(times)
    best = None
    for name, d in FC.items():
        gf = d["fit"]
        if not gf: continue
        span = d["info"]["gps_fit"]["span_s"]
        u0, u1 = gf["a"] + gf["b"] * span[0], gf["a"] + gf["b"] * span[1]
        overlap = max(0.0, min(u1, t1) - max(u0, t0))
        if overlap > 0 and (best is None or overlap > best[1]): best = (name, overlap)
    REPORT["pod_session_fc_log"][f"{node}/{sess}"] = dict(fc_log=best[0] if best else None, overlap_s=best[1] if best else 0.0,
                                                           exposures_utc=[t0, t1])
pd.DataFrame([dict(session=k, **v) for k, v in REPORT["pod_session_fc_log"].items()])


### Over the armed window only

The flight is the part that matters. Each `.bin`'s arm and disarm go to UTC through its GPS fit,
and every machine's pulse phase and chrony offset are summarised inside that window alone.

In [ ]:
armed = []
for name, d in FC.items():
    ev = d["info"].get("arm_events", []); gf = d["fit"]
    if not ev or not gf: continue
    t_arm = [e["t_s"] for e in ev if e["armed"]]; t_dis = [e["t_s"] for e in ev if not e["armed"]]
    if not t_arm: continue
    u0 = gf["a"] + gf["b"] * t_arm[0]; u1 = gf["a"] + gf["b"] * (min([t for t in t_dis if t > t_arm[0]], default=t_arm[0] + 1))
    REPORT["fc_logs"][name]["armed_utc"] = [u0, u1]
    for label, m in [(f"coordinator/{k[:8]}", v) for k, v in COORD.items()] + [(k, v) for k, v in PODS.items()]:
        if len(m["pps"]):
            ps = phase_stats(m["pps"], u0, u1)
            if ps["n"]: armed.append(dict(fc_log=name, machine=label, record="PPS phase", n=ps["n"], median_us=ps["median_us"], sd_us=ps["sd_us"], max_abs_us=ps["max_abs_us"], missing_s=ps["missing_seconds"]))
        for (metric, inst), s in m["ch"].items():
            if metric != "time_offset": continue
            w = s[(s.epoch >= u0) & (s.epoch <= u1)].dropna(subset=["value"])
            if len(w): armed.append(dict(fc_log=name, machine=label, record=f"chrony time_offset {inst}", n=int(len(w)), median_us=float(w.value.median() * 1e6), sd_us=float(w.value.std() * 1e6), max_abs_us=float(w.value.abs().max() * 1e6), missing_s=None))
armed = pd.DataFrame(armed); display(armed)
REPORT["armed_window"] = armed.to_dict("records")


## 5. The closure: a still's UTC label against GPS

A still's `exposure_wall_ns` is the pod's clock, disciplined to the chip through chrony. The same
instant can be put on the FC clock through `timesync.jsonl` and then onto GPS time through the
FC's GPS fit. The difference is how far the vehicle's UTC label sat from GPS during the flight.
The same quantity is measured two other ways: `fc_time.jsonl` (coordinator, through the FC) and
the FC's own `RTC` against its GPS. The three should agree to the ~19 ms floor the FC RTC has
against its GPS, and they are listed together.

In [ ]:
closure = []
for name, p in PAIR.items():
    if not p.get("session"): continue
    c = COORD[p["session"]]; ts = c["ts"]; grp = ts[(ts.boot == p["boot"]) & ts.steady]
    gf = FC[name]["fit"]
    # two routes from coordinator realtime to FC TimeUS: the FC's own requests (one-way delay in
    # the intercept) and the coordinator's requests in fc_time.jsonl (round trip halved)
    routes = []
    if len(grp) >= 3:
        routes.append(("timesync.jsonl", fit_line(grp.tc1_realtime_ns / 1e9, grp.fc_ts1_ns / 1e9, robust=True), grp.tc1_realtime_ns.min() / 1e9, grp.tc1_realtime_ns.max() / 1e9))
    ft = c["ft"]
    if p.get("fc_time") and len(ft):
        fg = ft[(ft.boot == p["fc_time"]["boot"]) & ft.steady]
        if len(fg) >= 3:
            routes.append(("fc_time.jsonl", fit_line(fg.local_us / 1e6, fg.fc_boot_us / 1e6, robust=True), fg.local_us.min() / 1e6, fg.local_us.max() / 1e6))
    for route, f_rt_to_fc, t0, t1 in routes:
        for node, pod in PODS.items():
            sc = pod["stills"]
            if not len(sc): continue
            rt = sc.exposure_wall_ns / 1e9
            rt = rt[(rt > t0 - 60) & (rt < t1 + 60)]
            if not len(rt): continue
            fc_s = f_rt_to_fc["a"] + f_rt_to_fc["b"] * rt
            utc_gps = gf["a"] + gf["b"] * fc_s
            off = (rt - utc_gps) * 1e3
            closure.append(dict(fc_log=name, pod=f"{node} stills via {route}", n=int(len(off)), label_minus_gps_ms=dict(median=float(np.median(off)), sd=float(np.std(off)))))
    # fc_time.jsonl: offset_us = FC UTC (its RTC) minus coordinator realtime. Coordinator minus GPS
    # is then -(offset) + (FC RTC minus GPS), the second term from the dataflash.
    # the coordinator's own label, the same way: its realtime at each fc_time.jsonl exchange against
    # GPS time at the FC monotonic that exchange measured; and the FC's two UTC labels against GPS
    if p.get("fc_time") and len(ft):
        fg = ft[(ft.boot == p["fc_time"]["boot"]) & ft.steady]
        if len(fg):
            off = (fg.local_us / 1e6 - (gf["a"] + gf["b"] * fg.fc_boot_us / 1e6)) * 1e3
            closure.append(dict(fc_log=name, pod="coordinator realtime, at fc_time.jsonl exchanges", n=int(len(fg)), label_minus_gps_ms=dict(median=float(off.median()), sd=float(off.std()))))
            off2 = (fg.fc_unix_us / 1e6 - (gf["a"] + gf["b"] * fg.fc_boot_us / 1e6)) * 1e3
            closure.append(dict(fc_log=name, pod="FC UTC as SYSTEM_TIME reports it", n=int(len(fg)), label_minus_gps_ms=dict(median=float(off2.median()), sd=float(off2.std()))))
    rg = FC[name]["info"].get("rtc_minus_gps_ms")
    if rg:
        closure.append(dict(fc_log=name, pod="FC UTC as the RTC rows log it", n=rg["n"], label_minus_gps_ms=dict(median=rg["median"], sd=None)))
closure = pd.DataFrame(closure); display(closure)
REPORT["closure"] = closure.to_dict("records")


## 6. Ground

The ground tlogs are stamped by cluster machines with NTP; `SYSTEM_TIME` inside them carries the
FC's UTC, so stamp minus `time_unix_usec` is the one-way path plus any label error. An empty or
unparseable file is reported as such.

In [ ]:
ground = []
for f in sorted(GND.glob("*.tlog*")) if GND.exists() else []:
    row = dict(file=f.name, bytes=f.stat().st_size)
    if f.stat().st_size == 0:
        row["note"] = "empty file"; ground.append(row); continue
    try:
        m = mavutil.mavlink_connection(str(f), robust_parsing=True)
        d = []
        while True:
            x = m.recv_match(type=["SYSTEM_TIME"], blocking=False)
            if x is None: break
            if x.time_unix_usec: d.append(m._timestamp - x.time_unix_usec / 1e6)
        row["n_system_time"] = len(d)
        if d: row["stamp_minus_fc_utc_ms"] = dict(median=float(np.median(d) * 1e3), sd=float(np.std(d) * 1e3))
    except Exception as e:
        row["note"] = f"unparseable: {e}"
    ground.append(row)
ground = pd.DataFrame(ground); display(ground)
REPORT["ground"] = ground.to_dict("records")


## 6b. Three observers of one link: what the FC sent, what each tlog received

The FC counts what it transmits per MAVLink channel (`MAV.txp`, cumulative per `chan`), the
coordinator records every frame it receives (`vehicle.tlog`, over the wired MAV2 serial), and the
ground station records what reached it over ELRS. Over the armed window, the FC's count for a
channel against the frames that channel's observer logged is that link's loss, measured rather
than inferred; the 8-bit sequence numbers inside each tlog give the same loss a second way.
Channels follow `SERIALn` order (USB, the coordinator on SERIAL4, ELRS on SERIAL6 --
`docs/flight-data-interpretation.md`); the mapping is stated here and checked against the counts,
not assumed by the code. `RC_CHANNELS` and `SERVO_OUTPUT_RAW` are listed with the rest, since
the interpretation doc records that stream as off. coordinator#229.

In [ ]:
observers = []; msg_tables = {}
for name, p in PAIR.items():
    au = REPORT["fc_logs"][name].get("armed_utc")
    gf = FC[name]["fit"]
    if not au or not gf: continue
    u0, u1 = au
    # the FC's own counters over the window, per channel
    mv = FC[name]["F"].get("MAV", pd.DataFrame())
    if len(mv):
        mv = mv.assign(utc=gf["a"] + gf["b"] * mv.TimeUS / 1e6)
        for chan, g in mv.groupby("chan"):
            w = g[(g.utc >= u0) & (g.utc <= u1)].sort_values("TimeUS")
            if len(w) >= 2:
                delta = lambda col: int((w[col].iloc[-1] - w[col].iloc[0]) % 65536)     # the counters are uint16
                observers.append(dict(fc_log=name, observer=f"FC MAV chan {int(chan)}", frames=delta("txp"),
                                      rx_frames=delta("rxp"), rx_dropped=delta("rxdp"),
                                      stream_slowdown=int(w.ss.iloc[-1]), seq_lost=None, bytes_skipped=None, n_types=None))
    # the coordinator's tlog for the paired session, and any ground tlog, over the same window
    files = []
    if p.get("session"):
        vt = CAP / "coordinator" / p["session"] / "vehicle.tlog"
        if vt.exists(): files.append(("coordinator vehicle.tlog", vt))
    if GND.exists():
        files += [(f"ground {f.name}", f) for f in sorted(GND.glob("*.tlog*"))]
    for label, f in files:
        if f.stat().st_size == 0 or not tlog_read.is_tlog(f):
            observers.append(dict(fc_log=name, observer=label, frames=0, note="empty file" if f.stat().st_size == 0 else "not a tlog")); continue
        df, info = tlog_read.read_tlog(f, u0, u1)
        fc_frames = df[df.sysid == 1] if len(df) else df
        loss = tlog_read.seq_loss(fc_frames)
        observers.append(dict(fc_log=name, observer=label, frames=int(len(fc_frames)), rx_frames=int(len(df) - len(fc_frames)),
                              seq_lost=int(loss.lost.sum()) if len(loss) else 0, bytes_skipped=info["bytes_skipped"],
                              n_types=int(fc_frames.msg.nunique()) if len(fc_frames) else 0))
        msg_tables[(name, label)] = tlog_read.message_table(fc_frames, u0, u1)
observers = pd.DataFrame(observers); display(observers)
for (name, label), t in msg_tables.items():
    print(f"\n{label} over the armed window of {name}: per message type (frames from the FC, sysid 1)")
    display(t)
REPORT["observers"] = observers.to_dict("records")
REPORT["observer_message_tables"] = {f"{k[0]}|{k[1]}": v.to_dict("records") for k, v in msg_tables.items()}


## 7. Summary

One row per pairing, with the figure `docs/time-base.md` and `docs/flight-data-interpretation.md`
give for it where one exists. The JSON beside this notebook carries every number above.

In [ ]:
rows = []
for name, d in FC.items():
    gf = d["info"].get("gps_fit")
    rows.append(dict(pairing=f"FC TimeUS vs GPS ({name})", n=gf and gf["n"], rate_ppm=gf and gf["rate_ppm"], rms=gf and f"{gf['rms_ms']:.1f} ms over {gf['span_s'][1]-gf['span_s'][0]:.0f} s", advertised="8.7-11.3 ms rms, slope per flight"))
    rows.append(dict(pairing=f"FC CAM edges ({name})", n=d["info"]["n_cam"], rate_ppm=(d["info"].get("cam_edges") or {}).get("fc_rate_vs_pulse_ppm"), rms=None, advertised="one row per PPS edge, hardware-captured"))
def lock_text(pp):
    lk = pp.get("lock") or {}
    if not lk.get("locked"): return f"never within {LOCK_US:.0f} us; max |phase| {lk.get('max_abs_us', 0)/1e3:.0f} ms"
    lp = pp.get("locked_phase", {})
    since = f"after a {lk['clock_steps_s'][-1]/86400:.0f}-day step" if lk.get("clock_steps_s") else "after the first assert"
    return f"locked {lk['s_to_lock']:.0f} s {since} (largest phase change {lk['max_phase_change_ms_per_s']:.0f} ms/s); then sd {lp.get('sd_us', 0):.1f} us, max {lp.get('max_abs_us', 0):.1f} us, {lp.get('missing_seconds', 0)} missing s"
def chrony_text(s):
    to = s.get("time_offset") or {}
    if not to.get("n"): return None
    return f"post-lock median {to['median']*1e6:.1f} us, max {to['max_abs']*1e6:.1f} us; whole session max {to['max_abs_all']*1e6:.0f} us"
for sname, c in COORD.items():
    pp = c["info"].get("pps")
    if pp: rows.append(dict(pairing=f"coordinator PPS phase ({sname[:8]})", n=pp["n"], rate_ppm=None, rms=lock_text(pp), advertised="0.3-3 us rms, zero gaps"))
    for b in c["info"]["timesync_boots"]:
        rows.append(dict(pairing=f"FC vs coordinator realtime, TIMESYNC ({sname[:8]} boot {b['boot']})", n=b["n_fit"], rate_ppm=b["fc_rate_vs_coord_ppm"], rms=b["rms_ms"] and f"{b['rms_ms']:.2f} ms ({b['n']} pairs, {b['n_steady']} after lock, {b['n_fit']} kept)", advertised="--"))
    for ftd in c["info"]["fc_time_boots"]:
        if ftd.get("fc_rate_vs_coord_ppm") is not None:
            rows.append(dict(pairing=f"FC vs coordinator realtime, fc_time.jsonl ({sname[:8]} boot {ftd['boot']})", n=ftd["n_steady"], rate_ppm=ftd["fc_rate_vs_coord_ppm"], rms=f"{ftd['fc_rate_fit_rms_ms']:.2f} ms, rtt median {ftd['rtt_ms']['median']:.1f} ms", advertised="--"))
            rows.append(dict(pairing=f"FC UTC (SYSTEM_TIME) minus coordinator realtime ({sname[:8]} boot {ftd['boot']})", n=ftd["n_steady"], rate_ppm=ftd["fc_minus_coord_drift_ppm"], rms=f"median {ftd['fc_minus_coord_ms']['median']:.1f} ms, fit rms {ftd['fit_rms_ms']:.1f} ms", advertised="--"))
    for inst, s in c["info"]["chrony"].items():
        t = chrony_text(s)
        if t: rows.append(dict(pairing=f"coordinator chrony: {'system' if inst == 'chrony' else 'source ' + inst}", n=s["time_offset"]["n"], rate_ppm=(s.get("clock_skew_ppm") or {}).get("median"), rms=t, advertised="PPS selected, stratum 1, RMS 0.3-3 us"))
for node, p in PODS.items():
    pp = p["info"].get("pps")
    if pp: rows.append(dict(pairing=f"{node} PPS phase", n=pp["n"], rate_ppm=None, rms=lock_text(pp) + f"; pre-lock phase bins (ms: count) {pp.get('prelock_phase_clusters_ms')}", advertised="--"))
    for inst, s in p["info"]["chrony"].items():
        t = chrony_text(s)
        if t: rows.append(dict(pairing=f"{node} chrony: {'system' if inst == 'chrony' else 'source ' + inst}", n=s["time_offset"]["n"], rate_ppm=(s.get("clock_skew_ppm") or {}).get("median"), rms=t, advertised="vs coordinator: RMS 26-31 us, stratum 2 (before the pod PPS refclock)"))
    fr = p["info"].get("frames")
    if fr: rows.append(dict(pairing=f"{node} exposure within its second", n=fr["n"], rate_ppm=None, rms=f"sd {fr['second_offset_us']['sd']:.0f} us, range {fr['second_offset_us']['min']:.0f}..{fr['second_offset_us']['max']:.0f} us", advertised="--"))
for r in REPORT.get("pod_pairs", []):
    if r.get("dt_us"): rows.append(dict(pairing=f"{r['a']} vs {r['b']} same-second exposures, nearest exposure_wall_ns", n=r["n_matched"], rate_ppm=None, rms=f"median {r['dt_us']['median']:.0f} us, sd {r['dt_us']['sd']:.0f} us, max {r['dt_us']['max_abs']:.0f} us", advertised="within 52 us at buffer_count=4"))
    if r.get("by_seq_frame_wall_clock_us"):
        q = r["by_seq_frame_wall_clock_us"]
        rows.append(dict(pairing=f"{r['a']} vs {r['b']} FrameWallClock by sequence number", n=q["n"], rate_ppm=None, rms=f"median {q['median']:.0f} us, sd {q['sd']:.0f} us, max {q['max_abs']:.0f} us", advertised="within 52 us at buffer_count=4"))
for r in REPORT.get("closure", []):
    rows.append(dict(pairing=f"UTC label minus GPS: {r['pod']} ({r['fc_log']})", n=r["n"], rate_ppm=None, rms=f"median {r['label_minus_gps_ms']['median']:.1f} ms", advertised="FC RTC vs GPS ~19 ms floor"))
for r in REPORT.get("observers", []):
    if r.get("observer", "").startswith("FC MAV"):
        rows.append(dict(pairing=f"{r['observer']} frames sent in the armed window ({r['fc_log']})", n=r["frames"], rate_ppm=None, rms=f"rx {r['rx_frames']}, rx dropped {r['rx_dropped']}, stream slowdown {r['stream_slowdown']}", advertised="chan 1 = coordinator (SERIAL4), chan 2 = ELRS (SERIAL6)"))
    else:
        rows.append(dict(pairing=f"{r['observer']} frames from the FC in the armed window ({r['fc_log']})", n=r["frames"], rate_ppm=None, rms=r.get("note") or f"seq-lost {r['seq_lost']}, {r['n_types']} message types, {r['bytes_skipped']} bytes skipped", advertised="--"))
for r in REPORT.get("fc_rate_triangle", []):
    if pd.notna(r.get("coord_vs_gps_ppm_via_timesync", None)):
        rows.append(dict(pairing=f"coordinator (pulse) vs GPS, by difference via timesync.jsonl ({r['fc_log']})", n=r.get("timesync_n"), rate_ppm=r["coord_vs_gps_ppm_via_timesync"], rms=f"timesync rms {r['timesync_rms_ms']:.2f} ms", advertised="DS3234: +/-3.5 ppm (datasheet)"))
    if pd.notna(r.get("coord_vs_gps_ppm_via_fc_time", None)):
        rows.append(dict(pairing=f"coordinator (pulse) vs GPS, by difference via fc_time.jsonl ({r['fc_log']})", n=None, rate_ppm=r["coord_vs_gps_ppm_via_fc_time"], rms=f"fc_time rms {r['fc_time_rms_ms']:.2f} ms", advertised="DS3234: +/-3.5 ppm (datasheet)"))
summary = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120); display(summary)
REPORT["summary"] = summary.to_dict("records")
(OUT / "time-sync.json").write_text(json.dumps(REPORT, indent=1, default=str))
print("->", OUT / "time-sync.json")
